# Token and Positional Embeddings

Neural networks work with continuous numbers, not discrete token IDs. Embeddings convert each token ID into a dense vector that the model can process.

**What we'll cover:**
- Token embeddings: Map each vocabulary word to a vector
- Positional embeddings: Encode where each token appears in the sequence
- Combining them into input embeddings

In [1]:
import os
import requests
import tiktoken
import torch
from torch.utils.data import Dataset, DataLoader

print(f"PyTorch version: {torch.__version__}")

PyTorch version: 2.12.0


## How Embedding Layers Work

An embedding layer is essentially a lookup table. You give it an integer ID, and it returns the corresponding vector.

Let's start with a tiny example.

In [2]:
# Small example: 6 tokens, 3-dimensional embeddings
vocab_size = 6
output_dim = 3

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

print("Embedding weights (the lookup table):")
print(embedding_layer.weight)

Embedding weights (the lookup table):
Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


In [3]:
# Look up token ID 3
print("Token ID 3 maps to:")
print(embedding_layer(torch.tensor([3])))

Token ID 3 maps to:
tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)


In [4]:
# Look up multiple tokens at once
input_ids = torch.tensor([2, 3, 5, 1])

print(f"Input IDs: {input_ids}")
print(f"\nEmbeddings:")
print(embedding_layer(input_ids))

Input IDs: tensor([2, 3, 5, 1])

Embeddings:
tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


Each row in the output corresponds to one input token. The embedding layer just looks up each ID in its weight matrix.

## Token Embeddings at Scale

For GPT-2, we have:
- Vocabulary size: 50,257 tokens
- Embedding dimension: 768 (for the small model)

That's about 38 million parameters just for the token embeddings.

In [5]:
# Real-world scale (but we'll use 256 dimensions to save memory)
vocab_size = 50257
output_dim = 256

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

print(f"Token embedding parameters: {vocab_size * output_dim:,}")

Token embedding parameters: 12,865,792


## Loading Data for Testing

Let's create a small batch of data to test our embeddings.

In [6]:
# Reuse the dataset code from the previous notebook
class GPTDataset(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1:i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


def create_dataloader(txt, batch_size=4, max_length=256, stride=128,
                      shuffle=True, drop_last=True, num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDataset(txt, tokenizer, max_length, stride)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,
                      drop_last=drop_last, num_workers=num_workers)

In [7]:
# Load text
if not os.path.exists("the-verdict.txt"):
    url = (
        "https://raw.githubusercontent.com/rasbt/"
        "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
        "the-verdict.txt"
    )
    response = requests.get(url, timeout=30)
    with open("the-verdict.txt", "wb") as f:
        f.write(response.content)

with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [8]:
max_length = 4
dataloader = create_dataloader(
    raw_text, batch_size=8, max_length=max_length,
    stride=max_length, shuffle=False
)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)

print(f"Input shape: {inputs.shape}")
print(f"Inputs:\n{inputs}")

Input shape: torch.Size([8, 4])
Inputs:
tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])


## Applying Token Embeddings

In [9]:
token_embeddings = token_embedding_layer(inputs)

print(f"Token embeddings shape: {token_embeddings.shape}")
print(f"\nDimensions: (batch_size, sequence_length, embedding_dim)")
print(f"            ({inputs.shape[0]}, {inputs.shape[1]}, {output_dim})")

Token embeddings shape: torch.Size([8, 4, 256])

Dimensions: (batch_size, sequence_length, embedding_dim)
            (8, 4, 256)


## Why Positional Embeddings?

Token embeddings alone don't tell the model where each token appears. "The cat sat" and "sat cat The" would have the same token embeddings (just in different order), but the model can't see that order.

Positional embeddings add position information. Position 0 gets one vector, position 1 gets another, and so on.

GPT-2 uses learned positional embeddings - the model figures out the best position vectors during training.

In [10]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

print(f"Positional embedding layer: {context_length} positions, {output_dim} dimensions")

Positional embedding layer: 4 positions, 256 dimensions


In [11]:
# Create position indices: [0, 1, 2, 3]
pos_embeddings = pos_embedding_layer(torch.arange(max_length))

print(f"Positional embeddings shape: {pos_embeddings.shape}")

Positional embeddings shape: torch.Size([4, 256])


## Combining Token and Positional Embeddings

The final input to the transformer is simply the sum of token and positional embeddings.

In [12]:
input_embeddings = token_embeddings + pos_embeddings

print(f"Input embeddings shape: {input_embeddings.shape}")

Input embeddings shape: torch.Size([8, 4, 256])


The shape is still `(batch_size, sequence_length, embedding_dim)`. Each token vector now contains both:
- What the token is (from token embedding)
- Where it appears (from positional embedding)

## Summary

The embedding pipeline:

1. **Token IDs** (integers) -> **Token Embeddings** (vectors representing word meaning)
2. **Position indices** (0, 1, 2, ...) -> **Positional Embeddings** (vectors representing position)
3. **Input Embeddings** = Token Embeddings + Positional Embeddings

These input embeddings are what the transformer layers will process.

Next steps: Self-attention - how the model learns relationships between tokens.